# Epileptic Seizure Detection Using Deep Learning
**Dataset:** UCI Epileptic Seizure Recognition  
**Models:** 1D CNN · LSTM · CNN-LSTM Hybrid  
**Task:** Binary Classification — Seizure vs Non-Seizure

## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc, ConfusionMatrixDisplay)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, LSTM, Dense,
                                      Dropout, Flatten, BatchNormalization, Input)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)

os.makedirs('model', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print('TensorFlow version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

## 2. Data Loading & Exploration

In [ ]:
df = pd.read_csv('data/Epileptic Seizure Recognition.csv')
print('Dataset shape:', df.shape)
df.head()

In [ ]:
print('Dataset Info:')
print(f'  Rows: {df.shape[0]}')
print(f'  Columns: {df.shape[1]}')
print(f'  Missing values: {df.isnull().sum().sum()}')
print()
print('Class Distribution:')
print(df['y'].value_counts().sort_index())
print()
print('Class Meaning:')
print('  1 = Seizure activity (epileptic)')
print('  2 = Tumour area (non-seizure)')
print('  3 = Healthy area (non-seizure)')
print('  4 = Eyes closed (non-seizure)')
print('  5 = Eyes open (non-seizure)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Class distribution bar chart
class_labels = ['Seizure\n(1)', 'Tumour\n(2)', 'Healthy\n(3)', 'Eyes\nClosed(4)', 'Eyes\nOpen(5)']
counts = df['y'].value_counts().sort_index()
axes[0].bar(class_labels, counts.values, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6'])
axes[0].set_title('Class Distribution (Original 5 Classes)', fontsize=13)
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Binary class distribution
binary_counts = [(df['y'] == 1).sum(), (df['y'] != 1).sum()]
axes[1].pie(binary_counts, labels=['Seizure', 'Non-Seizure'],
            colors=['#e74c3c', '#3498db'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Binary Class Distribution', fontsize=13)

plt.tight_layout()
plt.savefig('outputs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualize sample EEG signals
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

seizure_sample = df[df['y'] == 1].iloc[0, 1:-1].values
non_seizure_sample = df[df['y'] == 5].iloc[0, 1:-1].values

axes[0].plot(seizure_sample, color='#e74c3c', linewidth=0.8)
axes[0].set_title('EEG Signal — Seizure (Class 1)', fontsize=12)
axes[0].set_xlabel('Time Steps')
axes[0].set_ylabel('Amplitude')
axes[0].grid(alpha=0.3)

axes[1].plot(non_seizure_sample, color='#3498db', linewidth=0.8)
axes[1].set_title('EEG Signal — Non-Seizure (Class 5)', fontsize=12)
axes[1].set_xlabel('Time Steps')
axes[1].set_ylabel('Amplitude')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/sample_eeg_signals.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocessing

In [ ]:
# Drop ID column, separate features and labels
df_clean = df.drop(columns=['Unnamed'])
X = df_clean.drop(columns=['y']).values
y = (df_clean['y'] == 1).astype(int).values   # 1 = seizure, 0 = non-seizure

print('Features shape:', X.shape)
print('Labels shape:', y.shape)
print(f'Seizure samples: {y.sum()} ({y.mean()*100:.1f}%)')
print(f'Non-seizure samples: {(1-y).sum()} ({(1-y).mean()*100:.1f}%)')

In [ ]:
# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalize — fit only on training data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Reshape to (samples, timesteps, channels) for CNN/LSTM
X_train_3d = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_3d = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print('Train set:', X_train_3d.shape, '| Labels:', y_train.shape)
print('Test set: ', X_test_3d.shape, '| Labels:', y_test.shape)

## 4. Model Definitions

In [ ]:
def get_callbacks(model_name):
    return [
        EarlyStopping(monitor='val_accuracy', patience=10,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=5, min_lr=1e-6, verbose=1),
        ModelCheckpoint(f'model/{model_name}.h5', monitor='val_accuracy',
                        save_best_only=True, verbose=0)
    ]

### Model 1: 1D CNN

In [ ]:
def build_cnn(input_shape=(178, 1)):
    model = Sequential([
        Input(shape=input_shape),

        Conv1D(64, kernel_size=5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        Conv1D(128, kernel_size=5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        Conv1D(256, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ], name='1D_CNN')

    model.compile(optimizer=Adam(0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

cnn_model = build_cnn()
cnn_model.summary()

In [ ]:
print('Training 1D CNN...')
cnn_history = cnn_model.fit(
    X_train_3d, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.15,
    callbacks=get_callbacks('cnn_model'),
    verbose=1
)

### Model 2: LSTM

In [ ]:
def build_lstm(input_shape=(178, 1)):
    model = Sequential([
        Input(shape=input_shape),

        LSTM(128, return_sequences=True),
        Dropout(0.3),

        LSTM(64, return_sequences=False),
        Dropout(0.3),

        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ], name='LSTM')

    model.compile(optimizer=Adam(0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

lstm_model = build_lstm()
lstm_model.summary()

In [ ]:
print('Training LSTM...')
lstm_history = lstm_model.fit(
    X_train_3d, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.15,
    callbacks=get_callbacks('lstm_model'),
    verbose=1
)

### Model 3: CNN-LSTM Hybrid

In [ ]:
def build_cnn_lstm(input_shape=(178, 1)):
    model = Sequential([
        Input(shape=input_shape),

        # CNN blocks — extract local features
        Conv1D(64, kernel_size=5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        # LSTM layers — capture temporal dependencies
        LSTM(128, return_sequences=True),
        Dropout(0.3),
        LSTM(64, return_sequences=False),
        Dropout(0.3),

        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ], name='CNN_LSTM')

    model.compile(optimizer=Adam(0.001),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

cnn_lstm_model = build_cnn_lstm()
cnn_lstm_model.summary()

In [ ]:
print('Training CNN-LSTM Hybrid...')
cnn_lstm_history = cnn_lstm_model.fit(
    X_train_3d, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.15,
    callbacks=get_callbacks('cnn_lstm_model'),
    verbose=1
)

## 5. Training Curves

In [ ]:
def plot_history(histories, names):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    colors = ['#e74c3c', '#3498db', '#2ecc71']

    for hist, name, color in zip(histories, names, colors):
        axes[0].plot(hist.history['accuracy'], label=f'{name} Train', color=color, linewidth=1.5)
        axes[0].plot(hist.history['val_accuracy'], label=f'{name} Val',
                     color=color, linewidth=1.5, linestyle='--')

        axes[1].plot(hist.history['loss'], label=f'{name} Train', color=color, linewidth=1.5)
        axes[1].plot(hist.history['val_loss'], label=f'{name} Val',
                     color=color, linewidth=1.5, linestyle='--')

    axes[0].set_title('Model Accuracy', fontsize=13)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].set_title('Model Loss', fontsize=13)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('outputs/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history(
    [cnn_history, lstm_history, cnn_lstm_history],
    ['1D CNN', 'LSTM', 'CNN-LSTM']
)

## 6. Model Comparison

In [ ]:
models = {
    '1D CNN': cnn_model,
    'LSTM': lstm_model,
    'CNN-LSTM': cnn_lstm_model
}

results = {}
for name, model in models.items():
    loss, acc = model.evaluate(X_test_3d, y_test, verbose=0)
    y_pred_prob = model.predict(X_test_3d, verbose=0).ravel()
    y_pred = (y_pred_prob >= 0.5).astype(int)
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
    roc_auc = auc(fpr, tpr)
    results[name] = {
        'accuracy': acc,
        'loss': loss,
        'auc': roc_auc,
        'y_pred': y_pred,
        'y_pred_prob': y_pred_prob,
        'fpr': fpr,
        'tpr': tpr
    }
    print(f'{name:12s} → Accuracy: {acc*100:.2f}%  |  AUC: {roc_auc:.4f}  |  Loss: {loss:.4f}')

best_model_name = max(results, key=lambda k: results[k]['accuracy'])
print(f'\nBest Model: {best_model_name} ({results[best_model_name]["accuracy"]*100:.2f}%)')

In [ ]:
# Comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
model_names = list(results.keys())
accuracies = [results[m]['accuracy'] * 100 for m in model_names]
aucs = [results[m]['auc'] for m in model_names]
colors = ['#e74c3c', '#3498db', '#2ecc71']

bars = axes[0].bar(model_names, accuracies, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Test Accuracy Comparison', fontsize=13)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim([min(accuracies) - 5, 100])
axes[0].axhline(y=92, color='black', linestyle='--', linewidth=1, label='92% target')
axes[0].legend()
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{acc:.2f}%', ha='center', fontweight='bold')

bars2 = axes[1].bar(model_names, aucs, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('ROC-AUC Score Comparison', fontsize=13)
axes[1].set_ylabel('AUC Score')
axes[1].set_ylim([min(aucs) - 0.05, 1.0])
for bar, a in zip(bars2, aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{a:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Best Model — Detailed Evaluation

In [ ]:
best = results[best_model_name]
y_pred_best = best['y_pred']
y_prob_best = best['y_pred_prob']

print(f'Best Model: {best_model_name}')
print('=' * 50)
print(classification_report(y_test, y_pred_best,
                             target_names=['Non-Seizure', 'Seizure']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Non-Seizure', 'Seizure'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_model_name}', fontsize=13)

tn, fp, fn, tp = cm.ravel()
print(f'True Positives  (seizure correctly detected):    {tp}')
print(f'True Negatives  (non-seizure correctly rejected): {tn}')
print(f'False Positives (false alarms):                  {fp}')
print(f'False Negatives (missed seizures):               {fn}')

plt.tight_layout()
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curve
fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ['#e74c3c', '#3498db', '#2ecc71']

for (name, res), color in zip(results.items(), colors_roc):
    ax.plot(res['fpr'], res['tpr'], color=color, linewidth=2,
            label=f"{name} (AUC = {res['auc']:.4f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Best Model

In [ ]:
best_model_obj = models[best_model_name]
best_model_obj.save('model/seizure_model.h5')
print(f'Best model ({best_model_name}) saved to model/seizure_model.h5')

print('\nOutputs saved:')
for f in os.listdir('outputs'):
    print(f'  outputs/{f}')

## 9. Summary

In [ ]:
print('=' * 55)
print('        EPILEPTIC SEIZURE DETECTION — RESULTS')
print('=' * 55)
for name, res in results.items():
    marker = ' <<< BEST' if name == best_model_name else ''
    print(f'  {name:12s}  Accuracy: {res["accuracy"]*100:.2f}%   AUC: {res["auc"]:.4f}{marker}')
print('=' * 55)
print(f'  False Alarms (FP): {fp}')
print(f'  Missed Seizures (FN): {fn}')
print('=' * 55)